# Omnilex — BGE-M3 Session 4: Laws Eval + Courts Embed

**Task 13 Steps 1–3** (Session 4 of the BGE-M3 pipeline plan).

| Step | What | Time (T4) |
|------|------|-----------|
| 0 | Install deps | ~2 min |
| 1 | Laws embedding (176K rows) | ~5 min |
| 2 | Verify laws index | <1 min |
| 3 | Laws-only eval (10 queries, top-k 2/3/5/10) | ~1 min |
| — | Courts embedding (2.4M rows) | ~60-90 min — committed run |

**Settings:**
- GPU: T4 x1 (required)
- Internet: ON (required for BGE-M3 model download)
- `SKIP_LAWS_EMBED = True` → load index from `/kaggle/input/omnilex-bge-m3-laws/` instead of re-embedding
- `EMBED_COURTS = True` → run courts embedding at the end (committed run only)


In [ ]:
# TOKENIZERS_PARALLELISM must be set before any torch/transformers import
# to prevent tokenizer worker deadlocks when loading BGE-M3 on GPU.
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

!pip install "FlagEmbedding>=1.2" faiss-cpu "rank-bm25>=0.2.2" -q
import torch
print(f'torch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
import faiss
print(f'faiss {faiss.__version__}')

In [ ]:
from pathlib import Path
input_root = Path('/kaggle/input')
mounted = sorted(p.name for p in input_root.iterdir()) if input_root.exists() else []
print('Mounted at /kaggle/input:', mounted or '(nothing)')

In [ ]:
import shutil, sys, os
from pathlib import Path

CODE_DST = Path('/kaggle/working/omnilex')

# Locate code: prefer mounted dataset, fallback to kagglehub download
MOUNTED = Path('/kaggle/input/omnilex-retrieval-code')
if MOUNTED.exists() and any(MOUNTED.iterdir()):
    CODE_SRC = MOUNTED
    print(f'Using mounted dataset: {CODE_SRC}')
else:
    print('Code dataset not mounted — downloading via kagglehub...')
    import kagglehub
    CODE_SRC = Path(kagglehub.dataset_download('moeghri/omnilex-retrieval-code'))
    print(f'Downloaded to: {CODE_SRC}')

if CODE_DST.exists():
    shutil.rmtree(CODE_DST)
shutil.copytree(str(CODE_SRC), str(CODE_DST))

# Locate competition data
DATA_DIR = CODE_DST / 'data'
DATA_DIR.mkdir(exist_ok=True)
COMP_CANDIDATES = [
    Path('/kaggle/input/llm-agentic-legal-information-retrieval'),
    Path('/kaggle/input/competitions/llm-agentic-legal-information-retrieval'),
]
COMP_DIR = next((p for p in COMP_CANDIDATES if p.exists()), None)
if COMP_DIR is None:
    raise FileNotFoundError(
        'Competition data not found.\n'
        'Fix: click + Add Data in the sidebar, search for\n'
        '     llm-agentic-legal-information-retrieval, re-run this cell.'
    )
print(f'Competition data at: {COMP_DIR}')
for fname in ['laws_de.csv', 'court_considerations.csv', 'val.csv', 'test.csv']:
    src, dst = COMP_DIR / fname, DATA_DIR / fname
    if src.exists() and not dst.exists():
        dst.symlink_to(src)
        print(f'Linked {fname}')

# Both sys.path (this process) and PYTHONPATH (subprocesses) must point to src/
sys.path.insert(0, str(CODE_DST / 'src'))
os.environ['PYTHONPATH'] = str(CODE_DST / 'src')
os.chdir(str(CODE_DST))
print('Working dir:', os.getcwd())

In [ ]:
import csv
from omnilex.retrieval.models import BgeM3Embedder
from omnilex.retrieval.dense_index import DenseIndexBuilder, DenseIndex
from omnilex.retrieval.dense_retriever import DenseRetriever
from tqdm.auto import tqdm
print('Imports OK')

In [ ]:
# --- Task 13 Step 1: Build (or load) laws index ---
#
# SKIP_LAWS_EMBED = True  -> load pre-built index from /kaggle/input/omnilex-bge-m3-laws/
#                            Use this if you already published the laws index as a dataset.
# SKIP_LAWS_EMBED = False -> embed from scratch (~5 min on T4)
SKIP_LAWS_EMBED = False

LAWS_INDEX_DIR = Path('/kaggle/working/bge_m3/laws')
embedder = None

if SKIP_LAWS_EMBED:
    PRE_BUILT = Path('/kaggle/input/omnilex-bge-m3-laws')
    if not PRE_BUILT.exists():
        raise FileNotFoundError(
            f'{PRE_BUILT} not found.\n'
            'Either set SKIP_LAWS_EMBED=False to re-embed, or mount the\n'
            'omnilex-bge-m3-laws dataset via the sidebar.'
        )
    import shutil
    LAWS_INDEX_DIR.mkdir(parents=True, exist_ok=True)
    for f in PRE_BUILT.iterdir():
        dst = LAWS_INDEX_DIR / f.name
        if not dst.exists():
            shutil.copy2(str(f), str(dst))
    print(f'Loaded pre-built laws index from {PRE_BUILT}')
    print('Loading BGE-M3 model (needed for query encoding)...')
    embedder = BgeM3Embedder(model_name='BAAI/bge-m3')
else:
    laws_csv = Path('data/laws_de.csv')
    print(f'Loading {laws_csv}...')
    with open(laws_csv, encoding='utf-8') as f:
        laws_records = list(tqdm(csv.DictReader(f), total=175_933, desc='Loading laws CSV', unit=' rows'))
    print(f'Loaded {len(laws_records):,} records')

    print('Loading BGE-M3 model...')
    embedder = BgeM3Embedder(model_name='BAAI/bge-m3')

    builder = DenseIndexBuilder(embedder)
    print('Building laws index...')
    builder.build_from_records(
        records=laws_records,
        citation_field='citation',
        text_field='text',
        title_field='title',
        output_dir=LAWS_INDEX_DIR,
        batch_size=64,
    )
    print('Laws embedding complete.')

In [ ]:
# --- Task 13 Step 2: Verify laws index ---
laws_idx = DenseIndex.load(LAWS_INDEX_DIR)
print(f'Laws index: {len(laws_idx.metadata):,} passages, dim={laws_idx.index.d}')
print(f'First citation: {laws_idx.metadata[0]["citation_raw"]}')
print(f'Last citation:  {laws_idx.metadata[-1]["citation_raw"]}')
assert len(laws_idx.metadata) > 100_000, 'Index suspiciously small — check embedding step'
print('Laws index OK')

In [ ]:
# --- Task 13 Step 3: Laws-only evaluation at top-k 2/3/5/10 ---
import json
from omnilex.citations.normalizer import CitationNormalizer
from omnilex.evaluation.metrics import citation_f1, macro_f1

Path('/kaggle/working/results').mkdir(exist_ok=True)

normalizer = CitationNormalizer()
retriever = DenseRetriever(embedder=embedder, laws_index=laws_idx)

with open('data/val.csv', encoding='utf-8') as f:
    queries = list(csv.DictReader(f))
print(f'Evaluating on {len(queries)} queries')

top_ks = [2, 3, 5, 10]
all_gold = []
all_retrieved = {k: [] for k in top_ks}

for q in tqdm(queries, desc='Evaluating queries'):
    gold_raw = q.get('gold_citations', '')
    gold_canonical = normalizer.canonicalize_list([c.strip() for c in gold_raw.split(';') if c.strip()])
    all_gold.append(gold_canonical)
    candidates = retriever.retrieve(q['query'], top_k=max(top_ks), faiss_top_k=100)
    for k in top_ks:
        pred = normalizer.canonicalize_list([c.citation_raw for c in candidates[:k]])
        all_retrieved[k].append(pred)

results = {'val_queries': len(queries)}
for k in top_ks:
    scores = macro_f1(all_retrieved[k], all_gold)
    results[f'top_{k}'] = scores
    print(f'  top-{k}: Macro F1={scores["macro_f1"]:.4f}  P={scores["macro_precision"]:.4f}  R={scores["macro_recall"]:.4f}')

best_k = max(top_ks, key=lambda k: results[f'top_{k}']['macro_f1'])
results['best_k'] = best_k
results['per_query'] = []
for i, q in enumerate(queries):
    pq = citation_f1(all_retrieved[best_k][i], all_gold[i])
    pq['query_id'] = q['query_id']
    pq['query_preview'] = q['query'][:100]
    pq['num_gold'] = len(all_gold[i])
    pq['num_predicted'] = len(all_retrieved[best_k][i])
    results['per_query'].append(pq)

with open('/kaggle/working/results/bgem3_laws_only.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f'Saved. Best k={best_k} -> Macro F1={results[f"top_{best_k}"]["macro_f1"]:.4f}')

In [ ]:
import json
with open('/kaggle/working/results/bgem3_laws_only.json') as f:
    res = json.load(f)
print(f"\nLaws-only BGE-M3 ({res['val_queries']} val queries)")
print(f"{'top-k':<8} {'Macro F1':<12} {'Precision':<12} {'Recall'}")
print('-' * 44)
for k in [2, 3, 5, 10]:
    m = res.get(f'top_{k}', {})
    print(f"{k:<8} {m.get('macro_f1', 0):<12.4f} {m.get('macro_precision', 0):<12.4f} {m.get('macro_recall', 0):.4f}")
best = res['best_k']
print(f"\nBest k={best} -> Macro F1={res[f'top_{best}']['macro_f1']:.4f}")

In [ ]:
# --- Courts embedding (~60-90 min on T4) ---
#
# EMBED_COURTS = True  -> run courts embedding (committed run)
# EMBED_COURTS = False -> skip (interactive dev)
EMBED_COURTS = True

if EMBED_COURTS:
    courts_csv = Path('data/court_considerations.csv')
    print(f'Loading {courts_csv}...')
    with open(courts_csv, encoding='utf-8') as f:
        courts_records = list(tqdm(csv.DictReader(f), total=2_400_000, desc='Loading courts CSV', unit=' rows'))
    print(f'Loaded {len(courts_records):,} records')

    print('Building courts index...')
    builder_c = DenseIndexBuilder(embedder)
    builder_c.build_from_records(
        records=courts_records,
        citation_field='citation',
        text_field='text',
        title_field=None,
        output_dir=Path('/kaggle/working/bge_m3/courts'),
        batch_size=64,
    )
    courts_idx = DenseIndex.load('/kaggle/working/bge_m3/courts')
    print(f'Courts index: {len(courts_idx.metadata):,} passages, dim={courts_idx.index.d}')
    print('Courts embedding complete.')
else:
    print('Skipping courts (EMBED_COURTS=False).')

## After This Run

1. Copy `results/bgem3_laws_only.json` from the **Output** tab into
   `Omnilex-Agentic-Retrieval-Competition/results/` and commit:
   ```bash
   git add results/bgem3_laws_only.json
   git commit -m "results: BGE-M3 laws-only eval on val.csv"
   ```

2. From the **Output** tab, publish `bge_m3/` as a new Kaggle dataset
   named `omnilex-bge-m3-indices`. This lets Session 5 load both
   laws + courts without re-embedding.

3. For Session 5 (full hybrid eval after courts index is ready):
   ```python
   python scripts/run_evaluation.py \\
       --laws-index /kaggle/input/omnilex-bge-m3-indices/laws \\
       --courts-index /kaggle/input/omnilex-bge-m3-indices/courts \\
       --val-csv data/val.csv --top-k 2 3 5 10
   ```
